# Training Model Prediksi Kesesuaian Usaha Alumni

Notebook ini melatih model klasifikasi empat kelas untuk alumni berstatus **Wirausaha**:

- Sangat Sesuai
- Sesuai
- Kurang Sesuai
- Tidak Sesuai

> Catatan: dataset awal bersifat sintetis berbasis rubrik. Jangan menyebut hasil evaluasinya sebagai validasi terhadap kondisi alumni nyata sebelum label diperiksa oleh pakar atau guru produktif.


In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt


## 1. Memuat dataset

Ubah `DATASET_PATH` sesuai lokasi file di Google Drive atau komputer.


In [ ]:
# Untuk Google Colab, contoh:
# from google.colab import drive
# drive.mount('/content/gdrive')
# DATASET_PATH = '/content/gdrive/MyDrive/Uji Coba Model Skripsi/dataset_wirausaha_1500_multiclass.csv'

DATASET_PATH = 'dataset_wirausaha_1500_multiclass.csv'
df = pd.read_csv(DATASET_PATH)
print('Ukuran dataset:', df.shape)
display(df.head())
print(df['kesesuaian_usaha'].value_counts())


## 2. Menentukan fitur dan target

`status_tracer` tidak dipakai karena seluruh baris bernilai Wirausaha. `usaha_lokasi` juga tidak dipakai karena lokasi bebas tidak mengukur relevansi bidang usaha.


In [ ]:
feature_columns = [
    'jurusan',
    'nilai_ujikom',
    'nilai_kejuruan',
    'tempat_pkl_relevan',
    'ekskul_aktif',
    'usaha_jenis',
    'tingkat_relevansi_usaha',
    'usaha_lama_berjalan',
    'pendapatan',
]
target_column = 'kesesuaian_usaha'

X = df[feature_columns].copy()
y = df[target_column].copy()

categorical_features = ['jurusan', 'usaha_jenis']
numeric_features = [c for c in feature_columns if c not in categorical_features]


## 3. Pipeline preprocessing dan model

One-hot encoding dengan `handle_unknown="ignore"` lebih aman daripada `LabelEncoder` untuk fitur kategori karena kategori tidak dianggap memiliki urutan angka dan input baru tidak langsung menyebabkan error.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features),
    ]
)

model = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=1,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])


## 4. Split data secara stratified dan training


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Akurasi data uji: {accuracy * 100:.2f}%')
print(classification_report(y_test, y_pred, zero_division=0))


## 5. Confusion matrix


In [ ]:
labels = list(pipeline.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
print('Urutan kelas:', labels)
print(cm)

ConfusionMatrixDisplay(cm, display_labels=labels).plot(xticks_rotation=35)
plt.tight_layout()
plt.show()


## 6. Cross-validation

Cross-validation memberi gambaran yang lebih stabil dibanding satu pembagian train-test.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
)
print('Skor tiap fold:', cv_scores)
print(f'Rata-rata CV: {cv_scores.mean() * 100:.2f}% ± {cv_scores.std() * 100:.2f}%')


## 7. Contoh prediksi dan confidence

Confidence adalah probabilitas kelas tertinggi, bukan skor kesesuaian substantif.


In [ ]:
sample = X_test.head(5)
predictions = pipeline.predict(sample)
probabilities = pipeline.predict_proba(sample)

for i, (prediction, probability) in enumerate(zip(predictions, probabilities), start=1):
    print(f'Data {i}')
    print('Prediksi:', prediction)
    print('Confidence:', f'{probability.max() * 100:.2f}%')
    print(dict(zip(pipeline.classes_, probability.round(4))))
    print('-' * 40)


## 8. Menyimpan model dan metadata

Pipeline menyimpan preprocessing dan Random Forest sekaligus. API cukup memuat satu file model.


In [ ]:
MODEL_PATH = 'model_wirausaha_pipeline.pkl'
METADATA_PATH = 'metadata_wirausaha.json'

joblib.dump(pipeline, MODEL_PATH)

metadata = {
    'model_name': 'Tracer Study - Prediksi Kesesuaian Usaha',
    'version': '1.0.0',
    'feature_columns': feature_columns,
    'target_column': target_column,
    'classes': list(pipeline.classes_),
    'test_accuracy': round(float(accuracy), 4),
    'cv_accuracy_mean': round(float(cv_scores.mean()), 4),
    'cv_accuracy_std': round(float(cv_scores.std()), 4),
    'note': 'Dataset sintetis berbasis rubrik; validasi ulang dengan data alumni berlabel pakar.',
}

Path(METADATA_PATH).write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Model tersimpan:', MODEL_PATH)
print('Metadata tersimpan:', METADATA_PATH)
